In [ ]:
!unzip DM_data3.zip

串流輸出內容已截斷至最後 5000 行。
  inflating: DM_data3/train/User_033/06048.csv  
  inflating: DM_data3/train/User_033/06049.csv  
  inflating: DM_data3/train/User_033/06050.csv  
  inflating: DM_data3/train/User_033/06051.csv  
  inflating: DM_data3/train/User_033/06052.csv  
  inflating: DM_data3/train/User_033/06053.csv  
  inflating: DM_data3/train/User_033/06054.csv  
  inflating: DM_data3/train/User_033/06055.csv  
  inflating: DM_data3/train/User_033/06056.csv  
  inflating: DM_data3/train/User_033/06057.csv  
  inflating: DM_data3/train/User_033/06058.csv  
  inflating: DM_data3/train/User_033/06059.csv  
  inflating: DM_data3/train/User_033/06060.csv  
  inflating: DM_data3/train/User_033/06061.csv  
  inflating: DM_data3/train/User_033/06062.csv  
  inflating: DM_data3/train/User_033/06063.csv  
  inflating: DM_data3/train/User_033/06064.csv  
  inflating: DM_data3/train/User_033/06065.csv  
  inflating: DM_data3/train/User_033/06066.csv  
  inflating: DM_data3/train/User_033/06067.csv  

In [ ]:
import os
import glob
import random
import copy
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import f1_score, classification_report, confusion_matrix
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn


# =========================================================
# 0. Reproducibility
# =========================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


SEED = 42
set_seed(SEED)


# =========================================================
# 1. Config
# =========================================================
ROOT = "./DM_data3"
TRAIN_ROOT = os.path.join(ROOT, "train")
TEST_ROOT = os.path.join(ROOT, "test")

FEATURE_COLS = [
    "mean_x", "mean_y", "mean_z",
    "std_x", "std_y", "std_z"
]

NUM_CLASSES = 6
BATCH_SIZE = 32
NUM_EPOCHS = 60
VAL_RATIO = 0.2
NUM_WORKERS = 0

LR = 1e-3
WEIGHT_DECAY = 3e-4

# InceptionTime
DEPTH = 6
OUT_CHANNELS = 32
DROPOUT = 0.4

# Augmentation
GAUSSIAN_NOISE_STD = 0.03
TIME_MASK_PROB = 0.5
TIME_MASK_MAX_WIDTH = 30
TIME_SHIFT_PROB = 0.5
TIME_SHIFT_MAX = 20

# Early stopping
PATIENCE = 12

# SWA
USE_SWA = True
SWA_START = 20
SWA_LR = 5e-4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cuda


In [ ]:
# =========================================================
# 2. Collect files
# =========================================================
def collect_user_files(train_root):
    user_to_files = {}
    user_dirs = sorted(glob.glob(os.path.join(train_root, "User_*")))

    for user_dir in user_dirs:
        user_name = os.path.basename(user_dir)
        csv_files = sorted(glob.glob(os.path.join(user_dir, "*.csv")))
        if len(csv_files) > 0:
            user_to_files[user_name] = csv_files

    return user_to_files


def split_users(user_to_files, val_ratio=0.2, seed=42):
    users = sorted(list(user_to_files.keys()))
    rng = random.Random(seed)
    rng.shuffle(users)

    n_val = max(1, int(len(users) * val_ratio))
    val_users = users[:n_val]
    train_users = users[n_val:]

    train_files = []
    val_files = []

    for u in train_users:
        train_files.extend(user_to_files[u])

    for u in val_users:
        val_files.extend(user_to_files[u])

    return train_users, val_users, train_files, val_files


def collect_test_files(test_root):
    test_files = []
    user_dirs = sorted(glob.glob(os.path.join(test_root, "User_*")))

    for user_dir in user_dirs:
        test_files.extend(sorted(glob.glob(os.path.join(user_dir, "*.csv"))))

    return test_files


# =========================================================
# 3. Normalization from train only
# =========================================================
def compute_feature_stats(file_list, feature_cols):
    all_sum = np.zeros(len(feature_cols), dtype=np.float64)
    all_sq_sum = np.zeros(len(feature_cols), dtype=np.float64)
    total_count = 0

    for path in file_list:
        df = pd.read_csv(path)
        x = df[feature_cols].values.astype(np.float64)

        all_sum += x.sum(axis=0)
        all_sq_sum += (x ** 2).sum(axis=0)
        total_count += x.shape[0]

    mean = all_sum / total_count
    var = all_sq_sum / total_count - mean ** 2
    std = np.sqrt(np.maximum(var, 1e-12))

    return mean.astype(np.float32), std.astype(np.float32)


# =========================================================
# 4. Augmentation
# =========================================================
def apply_time_mask(x, max_width=30):
    x = x.copy()
    T = x.shape[0]
    width = np.random.randint(5, max_width + 1)
    start = np.random.randint(0, T - width + 1)
    x[start:start + width, :] = 0.0
    return x


def apply_time_shift(x, max_shift=20):
    shift = np.random.randint(-max_shift, max_shift + 1)
    return np.roll(x, shift=shift, axis=0).copy()


def apply_gaussian_noise(x, std=0.03):
    noise = np.random.normal(0, std, size=x.shape).astype(np.float32)
    return x + noise


In [ ]:
# =========================================================
# 5. Dataset
# =========================================================
class HARFileDataset(Dataset):
    def __init__(
        self,
        file_list,
        feature_cols,
        mean=None,
        std=None,
        is_test=False,
        augment=False
    ):
        self.file_list = file_list
        self.feature_cols = feature_cols
        self.mean = mean
        self.std = std
        self.is_test = is_test
        self.augment = augment

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        path = self.file_list[idx]
        df = pd.read_csv(path)

        x = df[self.feature_cols].values.astype(np.float32)  # (300, 6)

        if self.mean is not None and self.std is not None:
            x = (x - self.mean) / (self.std + 1e-8)

        if self.augment and not self.is_test:
            if np.random.rand() < TIME_SHIFT_PROB:
                x = apply_time_shift(x, max_shift=TIME_SHIFT_MAX)

            if np.random.rand() < TIME_MASK_PROB:
                x = apply_time_mask(x, max_width=TIME_MASK_MAX_WIDTH)

            x = apply_gaussian_noise(x, std=GAUSSIAN_NOISE_STD)

        x = torch.tensor(x, dtype=torch.float32)

        file_id = df["file_id"].iloc[0]

        if self.is_test:
            return x, file_id
        else:
            y = int(df["label"].iloc[0])
            y = torch.tensor(y, dtype=torch.long)
            return x, y


# =========================================================
# 6. InceptionTime Model
# =========================================================
class InceptionBlock1D(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels=32,
        bottleneck_channels=32,
        kernel_sizes=(9, 19, 39)
    ):
        super().__init__()

        if in_channels > 1:
            self.bottleneck = nn.Conv1d(
                in_channels,
                bottleneck_channels,
                kernel_size=1,
                bias=False
            )
            conv_in = bottleneck_channels
        else:
            self.bottleneck = nn.Identity()
            conv_in = in_channels

        self.conv1 = nn.Conv1d(
            conv_in,
            out_channels,
            kernel_size=kernel_sizes[0],
            padding=kernel_sizes[0] // 2,
            bias=False
        )

        self.conv2 = nn.Conv1d(
            conv_in,
            out_channels,
            kernel_size=kernel_sizes[1],
            padding=kernel_sizes[1] // 2,
            bias=False
        )

        self.conv3 = nn.Conv1d(
            conv_in,
            out_channels,
            kernel_size=kernel_sizes[2],
            padding=kernel_sizes[2] // 2,
            bias=False
        )

        self.maxpool = nn.MaxPool1d(kernel_size=3, stride=1, padding=1)
        self.pool_conv = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size=1,
            bias=False
        )

        self.bn = nn.BatchNorm1d(out_channels * 4)
        self.relu = nn.ReLU()

    def forward(self, x):
        z = self.bottleneck(x)

        x1 = self.conv1(z)
        x2 = self.conv2(z)
        x3 = self.conv3(z)

        x4 = self.maxpool(x)
        x4 = self.pool_conv(x4)

        out = torch.cat([x1, x2, x3, x4], dim=1)
        out = self.bn(out)
        out = self.relu(out)

        return out


class InceptionTimeClassifier(nn.Module):
    def __init__(
        self,
        input_dim=6,
        num_classes=6,
        depth=6,
        out_channels=32,
        dropout=0.4
    ):
        super().__init__()

        blocks = []
        in_ch = input_dim
        block_out_ch = out_channels * 4

        for _ in range(depth):
            blocks.append(
                InceptionBlock1D(
                    in_channels=in_ch,
                    out_channels=out_channels,
                    bottleneck_channels=32,
                    kernel_sizes=(9, 19, 39)
                )
            )
            in_ch = block_out_ch

        self.blocks = nn.Sequential(*blocks)
        self.pool = nn.AdaptiveAvgPool1d(1)

        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(block_out_ch, num_classes)
        )

    def forward(self, x):
        # x: (B, 300, 6)
        x = x.transpose(1, 2)      # (B, 6, 300)
        x = self.blocks(x)         # (B, C, 300)
        x = self.pool(x).squeeze(-1)
        logits = self.head(x)
        return logits


In [ ]:
# =========================================================
# 7. Evaluation using macro-F1
# =========================================================
def evaluate(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    total = 0
    correct = 0

    y_true = []
    y_pred = []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)

            logits = model(x)
            loss = criterion(logits, y)

            preds = logits.argmax(dim=1)

            total_loss += loss.item() * x.size(0)
            total += x.size(0)
            correct += (preds == y).sum().item()

            y_true.extend(y.cpu().numpy().tolist())
            y_pred.extend(preds.cpu().numpy().tolist())

    acc = correct / total
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    weighted_f1 = f1_score(y_true, y_pred, average="weighted")

    return total_loss / total, acc, macro_f1, weighted_f1, y_true, y_pred

In [ ]:
# =========================================================
# 8. Prepare data
# =========================================================
user_to_files = collect_user_files(TRAIN_ROOT)

train_users, val_users, train_files, val_files = split_users(
    user_to_files,
    val_ratio=VAL_RATIO,
    seed=SEED
)

print("Train users:", len(train_users))
print("Val users:", len(val_users))
print("Train files:", len(train_files))
print("Val files:", len(val_files))
print("Overlap users:", set(train_users) & set(val_users))

mean, std = compute_feature_stats(train_files, FEATURE_COLS)
print("Feature mean:", mean)
print("Feature std :", std)

train_dataset = HARFileDataset(
    train_files,
    feature_cols=FEATURE_COLS,
    mean=mean,
    std=std,
    is_test=False,
    augment=True
)

val_dataset = HARFileDataset(
    val_files,
    feature_cols=FEATURE_COLS,
    mean=mean,
    std=std,
    is_test=False,
    augment=False
)

test_files = collect_test_files(TEST_ROOT)

test_dataset = HARFileDataset(
    test_files,
    feature_cols=FEATURE_COLS,
    mean=mean,
    std=std,
    is_test=True,
    augment=False
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

Train users: 48
Val users: 12
Train files: 8882
Val files: 2138
Overlap users: set()
Feature mean: [-0.15609443  0.00869293  0.19610131  0.05111963  0.04390548  0.0467427 ]
Feature std : [0.6178634  0.46279782 0.57602924 0.10763319 0.10265689 0.09662282]


In [ ]:
# =========================================================
# 9. Model / Loss / Optimizer / Scheduler
# =========================================================
model = InceptionTimeClassifier(
    input_dim=6,
    num_classes=NUM_CLASSES,
    depth=DEPTH,
    out_channels=OUT_CHANNELS,
    dropout=DROPOUT
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=NUM_EPOCHS
)

if USE_SWA:
    swa_model = AveragedModel(model)
    swa_scheduler = SWALR(
        optimizer,
        swa_lr=SWA_LR
    )


# =========================================================
# 10. Training
# =========================================================
best_macro_f1 = 0.0
best_val_acc = 0.0
best_epoch = -1
best_path = "best_inception_aug_f1.pth"

patience_counter = 0

for epoch in range(NUM_EPOCHS):
    model.train()

    total_loss = 0.0
    total = 0
    correct = 0

    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        logits = model(x)
        loss = criterion(logits, y)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)

        optimizer.step()

        preds = logits.argmax(dim=1)
        total_loss += loss.item() * x.size(0)
        total += x.size(0)
        correct += (preds == y).sum().item()

    train_loss = total_loss / total
    train_acc = correct / total

    if USE_SWA and (epoch + 1) >= SWA_START:
        swa_model.update_parameters(model)
        swa_scheduler.step()
    else:
        scheduler.step()

    val_loss, val_acc, val_macro_f1, val_weighted_f1, y_true, y_pred = evaluate(
        model,
        val_loader,
        criterion,
        device
    )

    print(
        f"Epoch {epoch+1:02d}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | "
        f"Val Macro-F1: {val_macro_f1:.4f} | "
        f"Val Weighted-F1: {val_weighted_f1:.4f}"
    )

    if val_macro_f1 > best_macro_f1:
        best_macro_f1 = val_macro_f1
        best_val_acc = val_acc
        best_epoch = epoch + 1
        torch.save(model.state_dict(), best_path)
        patience_counter = 0
        print("Saved best model by Macro-F1.")

print(f"Best Epoch: {best_epoch}")
print(f"Best Val Acc: {best_val_acc:.4f}")
print(f"Best Val Macro-F1: {best_macro_f1:.4f}")


Epoch 01/60 | Train Loss: 0.6502 | Train Acc: 0.7792 | Val Loss: 0.6327 | Val Acc: 0.7853 | Val Macro-F1: 0.4928 | Val Weighted-F1: 0.7688
Saved best model by Macro-F1.
Epoch 02/60 | Train Loss: 0.5096 | Train Acc: 0.8291 | Val Loss: 0.5050 | Val Acc: 0.8349 | Val Macro-F1: 0.5927 | Val Weighted-F1: 0.8126
Saved best model by Macro-F1.
Epoch 03/60 | Train Loss: 0.4720 | Train Acc: 0.8401 | Val Loss: 0.5057 | Val Acc: 0.8316 | Val Macro-F1: 0.5570 | Val Weighted-F1: 0.8068
Epoch 04/60 | Train Loss: 0.4527 | Train Acc: 0.8481 | Val Loss: 0.4593 | Val Acc: 0.8447 | Val Macro-F1: 0.5959 | Val Weighted-F1: 0.8298
Saved best model by Macro-F1.
Epoch 05/60 | Train Loss: 0.4360 | Train Acc: 0.8521 | Val Loss: 0.4379 | Val Acc: 0.8592 | Val Macro-F1: 0.6016 | Val Weighted-F1: 0.8456
Saved best model by Macro-F1.
Epoch 06/60 | Train Loss: 0.4255 | Train Acc: 0.8587 | Val Loss: 0.4301 | Val Acc: 0.8597 | Val Macro-F1: 0.6600 | Val Weighted-F1: 0.8432
Saved best model by Macro-F1.
Epoch 07/60 | Tr

In [ ]:
# =========================================================
# 11. Optional SWA evaluation and saving
# =========================================================
if USE_SWA:
    print("\nUpdating BN for SWA model...")
    update_bn(train_loader, swa_model, device=device)

    swa_val_loss, swa_val_acc, swa_macro_f1, swa_weighted_f1, _, _ = evaluate(
        swa_model,
        val_loader,
        criterion,
        device
    )

    print(
        f"SWA Val Loss: {swa_val_loss:.4f} | "
        f"SWA Val Acc: {swa_val_acc:.4f} | "
        f"SWA Macro-F1: {swa_macro_f1:.4f} | "
        f"SWA Weighted-F1: {swa_weighted_f1:.4f}"
    )

    if swa_macro_f1 > best_macro_f1:
        torch.save(swa_model.module.state_dict(), best_path)
        best_macro_f1 = swa_macro_f1
        print("Saved SWA model as best.")



Updating BN for SWA model...
SWA Val Loss: 0.4350 | SWA Val Acc: 0.8751 | SWA Macro-F1: 0.7284 | SWA Weighted-F1: 0.8736


In [ ]:
# =========================================================
# 12. Load best model and print validation report
# =========================================================
model.load_state_dict(torch.load(best_path, map_location=device))
model.eval()

val_loss, val_acc, val_macro_f1, val_weighted_f1, y_true, y_pred = evaluate(
    model,
    val_loader,
    criterion,
    device
)

print("\nFinal Best Model Validation:")
print(f"Val Loss: {val_loss:.4f}")
print(f"Val Acc: {val_acc:.4f}")
print(f"Val Macro-F1: {val_macro_f1:.4f}")
print(f"Val Weighted-F1: {val_weighted_f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_true, y_pred, digits=4))

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))



Final Best Model Validation:
Val Loss: 0.4698
Val Acc: 0.8704
Val Macro-F1: 0.7566
Val Weighted-F1: 0.8688

Classification Report:
              precision    recall  f1-score   support

           0     0.9264    0.9663    0.9459       860
           1     0.8968    0.8472    0.8713       903
           2     0.4468    0.3281    0.3784        64
           3     0.7143    0.7721    0.7420       136
           4     0.9231    0.8000    0.8571        15
           5     0.7017    0.7937    0.7449       160

    accuracy                         0.8704      2138
   macro avg     0.7682    0.7512    0.7566      2138
weighted avg     0.8692    0.8704    0.8688      2138

Confusion Matrix:
[[831  24   1   2   0   2]
 [ 65 765  21  17   1  34]
 [  0  23  21  12   0   8]
 [  1  16   4 105   0  10]
 [  0   0   0   3  12   0]
 [  0  25   0   8   0 127]]


In [ ]:
# =========================================================
# 13. Test inference
# =========================================================
results = []

with torch.no_grad():
    for x, file_ids in test_loader:
        x = x.to(device)

        logits = model(x)
        preds = logits.argmax(dim=1).cpu().numpy()

        for fid, pred in zip(file_ids, preds):
            results.append([int(fid), int(pred)])


# =========================================================
# 14. Save submission
# =========================================================
sub = pd.DataFrame(results, columns=["Id", "Label"])
sub.to_csv("submission_inception_aug_swa.csv", index=False)

print(sub.head())
print("Saved submission_inception_aug_swa.csv")

      Id  Label
0  11021      0
1  11022      0
2  11023      0
3  11024      0
4  11025      0
Saved submission_inception_aug_swa.csv
